# AGC 2027 Katongan — OSM Road-Network Optimizer **v5.1 (502 FIX)**

This revised notebook handles `502 Bad Gateway` from overloaded Overpass servers. It uses **a compact bounding box**, provider fallback, and cached genuine OSM walking/driving graphs. No synthetic roads are created.

Upload the **v5.1 master ZIP**, not the older v5 ZIP. If a graph has already been downloaded, the script caches it for the next run **within the Colab session**.

OSM result is still a road-based *candidate-location scenario*, not verified land, truck operations, or water quality.

In [ ]:
%pip -q install osmnx==2.1.1 geopandas scipy networkx shapely pyproj pandas nbformat

In [ ]:
from google.colab import files
import zipfile, os, pathlib
print('Upload the AGC2027_Katongan_MASTER_SI_v5_1_OSM_FIXED.zip file.')
up = files.upload()
zipname = next((x for x in up if x.lower().endswith('.zip')), None)
if zipname is None: raise ValueError('Upload the master ZIP file')
folder = pathlib.Path('/content/katongan_master')
folder.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(zipname) as z: z.extractall(folder)
candidates = [x.parent.parent for x in folder.rglob('NETWORK_OSM/run_network_osm.py')]
if not candidates: raise FileNotFoundError('NETWORK_OSM/run_network_osm.py missing from uploaded ZIP')
root = candidates[0]
print('PROJECT ROOT:', root)
if not (root/'NETWORK_OSM'/'RETRY_PATCH_VERSION.json').exists():
    raise RuntimeError('Old master v5 detected. Upload the corrected v5.1 ZIP instead.')
print('502 patch version:', (root/'NETWORK_OSM'/'RETRY_PATCH_VERSION.json').read_text()[:220])

In [ ]:
import subprocess,sys
p=subprocess.run([sys.executable,str(root/'NETWORK_OSM'/'test_network_core.py')],text=True,capture_output=True)
print(p.stdout,p.stderr)
if p.returncode: raise RuntimeError('Offline optimizer unit test failed')
p=subprocess.run([sys.executable,str(root/'NETWORK_OSM'/'test_network_download.py')],text=True,capture_output=True)
print(p.stdout,p.stderr)
if p.returncode: raise RuntimeError('Overpass fallback/cache offline unit test failed')

## Fetch authentic OSM roads and compute network-based candidate sites

A `502` is **server capacity**, not evidence that Katongan has no roads. This corrected script tries `overpass.private.coffee` first and the official public Overpass server second, with bounded retries. The request is a rectangle rather than a very long polygon. If both providers fail, it stops safely; rerun when service resumes. If only the driving network failed, the walking graph will remain cached during this Colab session.

**No invented road or location accuracy values will be produced.**

In [ ]:
import subprocess, sys
command=[sys.executable,str(root/'NETWORK_OSM'/'run_network_osm.py'),
         '--project',str(root),'--radius','500','--target','0.95','--max-sites','30',
         '--bbox-buffer-m','1000',
         '--endpoints','https://overpass.private.coffee/api,https://overpass-api.de/api',
         '--attempts-per-endpoint','2','--retry-base-seconds','6',
         '--request-timeout','120']
print('RUN:', ' '.join(command),flush=True)
p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in p.stdout:
    print(line.rstrip(),flush=True)
returncode=p.wait()
if returncode:
    print('Overpass download failed after configured providers. No synthetic roads are produced.')
    print('A previously downloaded walking/drive graph remains cached in NETWORK_OSM/osm_graph_cache.')
    raise RuntimeError('Real OSM download failed; retry this cell later. Do not use Euclidean 95.84% as network coverage.')
print('OSM acquisition and optimization succeeded.')

In [ ]:
import json, pathlib
results=root/'NETWORK_OSM'/'results'
summary=json.loads((results/'NETWORK_MODEL_SUMMARY.json').read_text())
print('Sites selected:',summary['selected_site_count'])
print('Network coverage:',round(summary['coverage_network_pct'],2),'%')
print('95% target reached:',summary['target_reached'])
print('Previously reported 95.84% is EUCLIDEAN, not comparable as network coverage!')
print('Graph source providers:',summary['graph_provenance']['graphs'])
print('Network result files:',[p.name for p in results.iterdir()])

In [ ]:
import zipfile
from google.colab import files
out='/content/Katongan_OSM_Network_Results.zip'
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
    for f in results.rglob('*'):
        if f.is_file():z.write(f,f.relative_to(results.parent))
print('Download network outputs:')
files.download(out)

## Next release gate

The **road network model** can now be used for a provisional updated WebGIS, after a reviewer inspects network connectivity and the unusual snap distances. Before finalizing a **physical** installation: verify (i) road/truck access, (ii) land rights and appropriate plot, (iii) slope/hazard, (iv) external water supply and capacity, (v) sanitation/protection and relevant water quality testing. A network-optimized point is **not** automatically a feasible water-distribution site.

**Important:** output is not proof of access to private plots or a reliable water depot. Inspect facility space and road connectivity before proposing construction.

## Integrate the successful Colab output into the WebGIS

After downloading `Katongan_OSM_Network_Results.zip` to your computer, place it beside your extracted master folder. From inside that master folder run:

`python NETWORK_OSM/import_osm_results.py --project . --results-zip ../Katongan_OSM_Network_Results.zip`

The importer checks the real GeoJSONs, copies the network model to the WebGIS, and creates `GIS/Katongan_AGC2027_network_layers.gpkg`. Then run the local app and choose **Jaringan jalan OSM** in the Distribusi section. **Physical feasibility still requires separate checks**. If the network run fails, the original Euclidean scenario stays labelled as such.